In [ ]:
from getpass import getpass

admin_rdm_url = 'https://admin.bh.rdm.yzwlab.com/' #'https://admin.staging.rdm.example.com/'
rdm_url = 'https://bh.rdm.yzwlab.com/'

idp_name_integrated_admin = None
idp_username_integrated_admin = None
idp_password_integrated_admin = None

idp_name_quota_test_1 = None
idp_username_quota_test_1 = None
idp_password_quota_test_1 = None

# idp_username_quota_test_1 が所属する機関名（テスト中に機関ストレージをAmazon S3へ切り替え、終了時にNII Storageへ戻す）
target_organization = None
institution_default_max_quota = 5
# クォータ一律設定のシステム既定値（テスト終了時にこの値へ戻し、次回実行時も100→5への変更として検証できるようにする）
system_default_max_quota = 100
s3_access_key = None
s3_secret_key = None
s3_bucket = None

# Server Side Encryption: False = NO
s3_server_side_encryption = False

# idp_username_quota_test_1 でログインした際にIdPから返されるメールアドレス（Adminのユーザ検索に使用）
email_search = None
default_result_path = None
close_on_fail = False
transition_timeout = 60000

In [ ]:
if idp_name_integrated_admin is None:
    idp_name_integrated_admin = input(prompt='IdP name for integrated_admin')
if idp_username_integrated_admin is None:
    idp_username_integrated_admin = input(prompt=f'Username for {idp_name_integrated_admin}')
if idp_password_integrated_admin is None:
    idp_password_integrated_admin = getpass(prompt=f'Password for {idp_username_integrated_admin}@{idp_name_integrated_admin}')
(len(idp_username_integrated_admin), len(idp_password_integrated_admin))

In [ ]:
if s3_access_key is None:
    s3_access_key = input(prompt='S3 Access Key')
if s3_secret_key is None:
    s3_secret_key = getpass(prompt='S3 Secret Key')
if s3_bucket is None:
    s3_bucket = input(prompt='S3 Bucket Name')
(len(s3_access_key), len(s3_secret_key), len(s3_bucket))

In [ ]:
if idp_name_quota_test_1 is None:
    idp_name_quota_test_1 = input(prompt='IdP name for quota_test_1')
if idp_username_quota_test_1 is None:
    idp_username_quota_test_1 = input(prompt=f'Username for {idp_name_quota_test_1}')
if idp_password_quota_test_1 is None:
    idp_password_quota_test_1 = getpass(prompt=f'Password for {idp_username_quota_test_1}@{idp_name_quota_test_1}')
if email_search is None:
    email_search = input(prompt='Email address of quota test user 1 (Admin ユーザ検索欄の入力値)')
if target_organization is None:
    target_organization = input(prompt='Target organization name for quota test (対象機関名)')
(len(idp_username_quota_test_1), len(idp_password_quota_test_1))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# 再ログイン_Institutions

- サブシステム名: 再ログインサイクル -> delete
- ページ/アドオン: ログイン
- 機能分類: 再ログインサイクル -> delete
- シナリオ名: 再ログインサイクル -> delete
- 用意するテストデータ: URL一覧、アカウント(クォータテストユーザー1), アカウント(統合管理者)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの同一ウィンドウでGakunin RDM管理者のトップページを表示する

管理者トップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=30000)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

個人の管理者ページが表示されること

In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin, idp_password_integrated_admin, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## サイドメニューの「機関ストレージ」を選択する

機関のリスト画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/custom_storage_location/institutional_storage/institutions/"]').click()

    await expect(page.locator('//h2[text() = "機関のリスト"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 機関のリストから対象機関を選択する

対象機関の機関ストレージ設定画面が表示されること

In [ ]:
import traceback

async def _step(page):
    while True:
        link = page.locator(f'//a[normalize-space() = "{target_organization}"]').last
        try:
            await expect(link).to_be_visible()
        except:
            traceback.print_exc()
            print('Search next page...')
            # 次のページかもしれない
            await page.locator('//a[i[contains(@class, "fa-angle-right")]]').click()
            await expect(page.locator('//h2[text() = "機関のリスト"]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break

    await expect(page.locator('#s3')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「Amazon S3」のラジオボタンを選択する

「Amazon S3」のラジオボタンが選択された状態になること

In [ ]:
import asyncio

async def _step(page):
    await page.locator('#s3').check()

    # #s3 の change イベントで storage_name の enable/disable が切り替わるため、
    # 状態確認前に少し待って反映されるのを待つ
    await asyncio.sleep(0.5)

    # storage_name フィールドの状態を確認し、必要なら入力する
    storage_name = page.locator('#storage_name')
    sn_visible = await storage_name.is_visible()
    sn_enabled = await storage_name.is_enabled() if sn_visible else False
    sn_value = await storage_name.input_value() if sn_visible else ''
    print(f'storage_name: visible={sn_visible}, enabled={sn_enabled}, value="{sn_value}"')
    if sn_visible and sn_enabled and not sn_value.strip():
        await storage_name.fill('Amazon S3')
        print('Filled storage_name: Amazon S3')

    await expect(page.locator('#s3')).to_be_checked()

await run_pw(_step)

## 「保存」ボタンをクリックし、確認ダイアログで確認文字列を入力して「変更」をクリックする

機関ストレージ変更の確認ダイアログ「機関のストレージを変更してもよろしいですか？」が表示される。表示された確認文字列を入力し、「変更」をクリックすると「Amazon S3アカウントに接続」モーダルが表示されること。

※ 確認文字列はランダムに生成されるため、画面を確認して手入力する。

In [ ]:
import asyncio

async def _step(page):
    # Save(保存) ボタンをクリック
    save_btn = page.locator('#institutional_storage_form button.next-btn[type="submit"]').first
    await save_btn.scroll_into_view_if_needed()
    await save_btn.click()
    await asyncio.sleep(2)

    # bootbox確認ダイアログが表示されるか確認
    confirm_text = page.locator('#bbConfirmText')
    s3_modal = page.locator('#s3_modal')

    try:
        await expect(confirm_text).to_be_visible(timeout=10000)
        # bootbox確認ダイアログが表示された
        confirm_strong = page.locator('//div[contains(@class, "bootbox-body")]//strong')
        confirmation_string = await confirm_strong.text_content()
        print(f'Confirmation string: {confirmation_string}')
        await confirm_text.fill(confirmation_string)
        await page.get_by_role('button', name='変更').click()
        await expect(s3_modal).to_be_visible(timeout=transition_timeout)
        print('Confirmed via bootbox dialog')
    except Exception:
        # bootbox確認がない場合、S3モーダルを待つ
        try:
            await expect(s3_modal).to_be_visible(timeout=transition_timeout)
            print('S3 modal appeared directly')
        except Exception as e:
            # どちらも表示されない場合、デバッグ情報を出力（invalidなフィールドを特定するため）
            debug_body = await page.evaluate('() => document.body.innerText.substring(0, 2000)')
            debug_modals = await page.evaluate('() => document.querySelectorAll(".modal.show, .modal.in").length')
            debug_bootbox = await page.evaluate('() => document.querySelectorAll(".bootbox").length')
            debug_validity = await page.evaluate('() => { var f = document.getElementById("institutional_storage_form"); return f ? f.checkValidity() : "form not found"; }')
            debug_invalid = await page.evaluate('''() => {
                var f = document.getElementById("institutional_storage_form");
                if (!f) return [];
                return Array.from(f.querySelectorAll(":invalid")).map(el => ({
                    tag: el.tagName, id: el.id, name: el.name, required: el.required,
                    value: el.value, visible: el.offsetParent !== null
                }));
            }''')
            debug_selected = await page.evaluate('() => { var r = document.querySelector("input[name=options]:checked"); return r ? r.value : "none"; }')
            raise Exception(
                f'Neither bootbox nor modal visible. '
                f'modals_show={debug_modals}, bootbox={debug_bootbox}, '
                f'checkValidity={debug_validity}, invalid_fields={debug_invalid}, '
                f'selectedProvider={debug_selected}, '
                f'body={debug_body[:500]}'
            ) from e

await run_pw(_step)

## 「アクセスキー」「シークレットキー」「バケット」欄を入力し、「サーバー側の暗号化を有効にする」の「有効化する」のチェックを外した状態にする

各項目が正しく入力されること

In [ ]:
import asyncio

async def _step(page):
    # Access Key を入力
    await page.locator('#s3_access_key').fill(s3_access_key)

    # Secret Key を入力
    await page.locator('#s3_secret_key').fill(s3_secret_key)

    # Bucket を入力
    await page.locator('#s3_bucket').fill(s3_bucket)

    # Server Side Encryption のチェックボックス
    sse_checkbox = page.locator('#s3_server_side_encryption')
    is_checked = await sse_checkbox.is_checked()
    if s3_server_side_encryption and not is_checked:
        await sse_checkbox.click()
    elif not s3_server_side_encryption and is_checked:
        await sse_checkbox.click()

    # keyupイベントを発火させてバリデーションをトリガー
    await page.evaluate('() => { document.querySelectorAll("#s3_modal input").forEach(el => el.dispatchEvent(new Event("keyup", { bubbles: true }))); }')

    # 入力後少し待つ
    await asyncio.sleep(1)

await run_pw(_step)

## 「接続」ボタンをクリックして接続テストを行う

接続テストが成功すること

In [ ]:
async def _step(page):
    # Connect ボタンをクリック
    connect_btn = page.locator('#s3_connect')
    await expect(connect_btn).to_be_enabled(timeout=transition_timeout)
    await connect_btn.click()

    # Save ボタンが有効になるのを待つ（接続成功の証）
    save_btn = page.locator('#s3_save')
    await expect(save_btn).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「保存」ボタンをクリックして設定を保存する

対象機関が「Amazon S3」を機関ストレージとして利用する状態になること

In [ ]:
async def _step(page):
    # Save ボタンをクリック
    save_btn = page.locator('#s3_save')
    await save_btn.click()

    # 保存成功メッセージは一瞬しか表示されずすぐページがリロードされるため、
    # モーダルが閉じたこと→ページ安定→ラジオの最終状態で完了を確認する
    s3_modal = page.locator('#s3_modal')
    await expect(s3_modal).to_be_hidden(timeout=transition_timeout)
    await page.wait_for_load_state('load', timeout=transition_timeout)
    await expect(page.locator('#s3')).to_be_checked(timeout=transition_timeout)

await run_pw(_step)

## 「機関ストレージのクォータ」を選択し、対象機関を選択する

「機関ストレージのクォータ制御」画面が表示されること

In [ ]:
import traceback

async def _step(page):
    await page.locator('//a[@href = "/institutional_storage_quota_control/"]').click()

    while True:
        link = page.locator(f'//a[normalize-space() = "{target_organization}"]').last
        try:
            await expect(link).to_be_visible()
        except:
            traceback.print_exc()
            print('Search next page...')
            # 次のページかもしれない
            await page.locator('//a[i[contains(@class, "fa-angle-right")]]').click()
            await expect(page.locator('//h2[text() = "機関ストレージ"]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break

    await expect(page.locator('#title')).to_contain_text('機関ストレージのクォータ制御', timeout=transition_timeout)

await run_pw(_step)

## 「クォータ一律設定 (GB)」欄に、システム既定値(100)と異なる任意の値（例：5）を入力し、「適用」ボタンを押下する

設定値が保存されること（画面に入力値が反映される）

In [ ]:
async def _step(page):
    await page.locator('#storageLimit').fill(str(institution_default_max_quota))
    await page.locator('#institutional_storage_form button[type="submit"]').click()

    await expect(page.locator('#storageLimit')).to_have_value(str(institution_default_max_quota), timeout=transition_timeout)

await run_pw(_step)

## 同一ウィンドウでGRDMトップページへ遷移する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する ボタンが現れるまで待つ
    await expect(page.locator('//button[text() = "同意する"]')).to_be_visible(timeout=transition_timeout)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## 「RCOS IdP」を利用し、クォータテストユーザー1としてログインする

設定画面が表示されること

In [ ]:
async def _step(page):
    await scripts.grdm.login(
        page, idp_name_quota_test_1, idp_username_quota_test_1, idp_password_quota_test_1, transition_timeout=transition_timeout
    )

    await expect(page.locator('h2.page-header')).to_contain_text('設定', timeout=transition_timeout)

await run_pw(_step)

※ (補足)ここでは新規ユーザーのUserQuotaに関する手順を実施します

※ (補足)新規ユーザーは初回ログイン時点でプロフィールの必須項目（「姓」「名前」「姓（英語）」「名前（英語）」等）が未入力であるため、ダッシュボードではなく「設定」画面へ遷移する。これは意図した挙動である。

## 同一ウィンドウでGakunin RDM管理者のページへ再度アクセスする

管理者としてログイン済みの状態が維持されており、個人の管理者ページが表示されること

In [ ]:
async def _step(page):
    await page.goto(admin_rdm_url)

    # GRDMトップページへの遷移およびクォータテストユーザー1のログインでは管理者をログアウトしていないため、
    # 管理者セッションは維持されたままであり、個人の管理者ページ（ダッシュボード）が表示されることを確認する
    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「機関ストレージのクォータ」から対象機関の「機関ストレージのクォータ制御」画面を表示し、一覧から新規作成されたクォータテストユーザー1の行を確認する

該当のユーザーが一覧に表示され、「クォータ」列が「クォータ一律設定 (GB)」に設定した値（例：5 GB）と一致することを確認する

In [ ]:
import traceback

async def _step(page):
    await page.locator('//a[@href = "/institutional_storage_quota_control/"]').click()

    while True:
        link = page.locator(f'//a[normalize-space() = "{target_organization}"]').last
        try:
            await expect(link).to_be_visible()
        except:
            traceback.print_exc()
            print('Search next page...')
            # 次のページかもしれない
            await page.locator('//a[i[contains(@class, "fa-angle-right")]]').click()
            await expect(page.locator('//h2[text() = "機関ストレージ"]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break

    while True:
        row = page.locator(f'//tr[td[contains(text(), "{email_search}")]]')
        try:
            await expect(row).to_be_visible(timeout=5000)
        except Exception:
            next_link = page.locator('//a[i[contains(@class, "fa-angle-right")]]')
            if await next_link.count() == 0:
                raise
            traceback.print_exc()
            print('Search next page for user row...')
            await next_link.click()
            continue
        break

    # スクリーンショットに対象行が写るよう、表示領域内へスクロールしておく
    await row.scroll_into_view_if_needed()

    await expect(row.locator('td').last).to_contain_text(f'{institution_default_max_quota} GB', timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ管理」から「ユーザ管理」選択し、 ページを表示する

ユーザ検索画面が表示される

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapseUsers"]').click()
    await page.locator('//a[@href = "/users/"]').click()

    await expect(page.locator('//input[@name = "guid"]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ検索」画面のeメール欄へメールアドレスを入力し、「検索」ボタンを押下する

該当のユーザーが表示される

In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(email_search)
    await page.locator('//input[@type = "submit"]').click()

    await expect(page.locator(f'//td[contains(text(), "{email_search}")]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ詳細」画面で「機関ストレージの割当て(GB)」欄の値を確認する

値が「クォータ一律設定 (GB)」に設定した値（例：5）と一致することを確認する

In [ ]:
async def _step(page):
    storage_limit = page.locator('#storageLimit')

    # スクリーンショットに対象欄が写るよう、表示領域内へスクロールしておく
    await storage_limit.scroll_into_view_if_needed()

    await expect(storage_limit).to_have_value(str(institution_default_max_quota), timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ詳細」画面で「GDPRアカウントの削除」ボタンをクリックする

「このユーザーをGDPR削除してもよろしいですか？」ダイアログが表示されること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="GDPRアカウントの削除").click()
    await expect( page.locator("#deleteModal h3")).to_contain_text("このユーザーをGDPR削除してもよろしいですか？")

await run_pw(_step)

## 「確認」ボタンをクリックする

「User <uid> was successfully GDPR deleted」メッセージが表示されること

In [ ]:
async def _step(page):
    uid = page.url.rstrip("/").split("/")[-1]
    print(uid)

    await page.locator('#deleteModal input[type="submit"][value="確認"]').click()
    await expect(page.get_by_role("alert").first).to_contain_text(f"User {uid} was successfully GDPR deleted")

await run_pw(_step)

## 「ユーザ管理」から「ユーザ管理」選択し、 ページを表示する

ユーザ検索画面が表示される

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "#collapseUsers"]').click()
    await page.get_by_role("link", name="ユーザ管理").click()

    await expect(page.locator('//input[@name = "guid"]').first).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ユーザ検索」画面のeメール欄へメールアドレスを入力し、「検索」ボタンを押下する

「User with email address <email_test> not found.」が表示されること

In [ ]:
async def _step(page):
    await page.locator('//input[@name = "email"]').fill(email_search)
    await page.locator('//input[@type = "submit"]').click()

    await expect(page.get_by_text(f'User with email address {email_search} not found.')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「機関ストレージのクォータ」から対象機関の「機関ストレージのクォータ制御」画面を表示し、「クォータ一律設定 (GB)」欄をシステム既定値(100)に戻し、「適用」ボタンを押下する

設定値がシステム既定値(100)に戻ること（画面に入力値が反映される）。次回テスト実行時にも100→5への変更として検証できるよう、S3からNII Storageへ切り替える前に既定値へ戻しておく。

In [ ]:
import traceback

async def _step(page):
    await page.locator('//a[@href = "/institutional_storage_quota_control/"]').click()

    while True:
        link = page.locator(f'//a[normalize-space() = "{target_organization}"]').last
        try:
            await expect(link).to_be_visible()
        except:
            traceback.print_exc()
            print('Search next page...')
            # 次のページかもしれない
            await page.locator('//a[i[contains(@class, "fa-angle-right")]]').click()
            await expect(page.locator('//h2[text() = "機関ストレージ"]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break

    storage_limit = page.locator('#storageLimit')
    await storage_limit.fill(str(system_default_max_quota))
    await page.locator('#institutional_storage_form button[type="submit"]').click()

    # スクリーンショットに対象欄が写るよう、表示領域内へスクロールしておく
    await storage_limit.scroll_into_view_if_needed()

    await expect(storage_limit).to_have_value(str(system_default_max_quota), timeout=transition_timeout)

await run_pw(_step)

## サイドメニューの「機関ストレージ」を選択する

機関のリスト画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[@href = "/custom_storage_location/institutional_storage/institutions/"]').click()

    await expect(page.locator('//h2[text() = "機関のリスト"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 機関のリストから対象機関を選択する

対象機関の機関ストレージ設定画面が表示されること

In [ ]:
import traceback

async def _step(page):
    while True:
        link = page.locator(f'//a[normalize-space() = "{target_organization}"]').last
        try:
            await expect(link).to_be_visible()
        except:
            traceback.print_exc()
            print('Search next page...')
            # 次のページかもしれない
            await page.locator('//a[i[contains(@class, "fa-angle-right")]]').click()
            await expect(page.locator('//h2[text() = "機関のリスト"]')).to_be_visible(timeout=transition_timeout)
            continue
        await link.click()
        break

    await expect(page.locator('#osfstorage')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「NII Storage」のラジオボタンを選択する

「NII Storage」のラジオボタンが選択された状態になること

In [ ]:
import asyncio

async def _step(page):
    await page.locator('#osfstorage').check()

    # #osfstorage の change イベントで storage_name の enable/disable が切り替わるため、
    # 状態確認前に少し待って反映されるのを待つ
    await asyncio.sleep(0.5)

    # storage_name フィールドの状態を確認する（NII Storageでは編集不可のはず）
    storage_name = page.locator('#storage_name')
    sn_visible = await storage_name.is_visible()
    sn_enabled = await storage_name.is_enabled() if sn_visible else False
    sn_value = await storage_name.input_value() if sn_visible else ''
    print(f'storage_name: visible={sn_visible}, enabled={sn_enabled}, value="{sn_value}"')

    await expect(page.locator('#osfstorage')).to_be_checked()

await run_pw(_step)

## 「保存」ボタンをクリックし、確認ダイアログで確認文字列を入力して「変更」をクリックする

機関ストレージ変更の確認ダイアログ「機関のストレージを変更してもよろしいですか？」が表示される。表示された確認文字列を入力し、「変更」をクリックすると「NIIストレージに接続」モーダルが表示されること。

※ 確認文字列はランダムに生成されるため、画面を確認して手入力する。

In [ ]:
import asyncio

async def _step(page):
    # Save(保存) ボタンをクリック
    save_btn = page.locator('#institutional_storage_form button.next-btn[type="submit"]').first
    await save_btn.scroll_into_view_if_needed()
    await save_btn.click()
    await asyncio.sleep(2)

    # bootbox確認ダイアログが表示されるか確認
    confirm_text = page.locator('#bbConfirmText')
    osfstorage_modal = page.locator('#osfstorage_modal')

    try:
        await expect(confirm_text).to_be_visible(timeout=10000)
        # bootbox確認ダイアログが表示された
        confirm_strong = page.locator('//div[contains(@class, "bootbox-body")]//strong')
        confirmation_string = await confirm_strong.text_content()
        print(f'Confirmation string: {confirmation_string}')
        await confirm_text.fill(confirmation_string)
        await page.get_by_role('button', name='変更').click()
        await expect(osfstorage_modal).to_be_visible(timeout=transition_timeout)
        print('Confirmed via bootbox dialog')
    except Exception:
        # bootbox確認がない場合、NIIストレージモーダルを待つ
        try:
            await expect(osfstorage_modal).to_be_visible(timeout=transition_timeout)
            print('NII Storage modal appeared directly')
        except Exception as e:
            # どちらも表示されない場合、デバッグ情報を出力
            debug_body = await page.evaluate('() => document.body.innerText.substring(0, 2000)')
            debug_modals = await page.evaluate('() => document.querySelectorAll(".modal.show, .modal.in").length')
            debug_bootbox = await page.evaluate('() => document.querySelectorAll(".bootbox").length')
            debug_validity = await page.evaluate('() => { var f = document.getElementById("institutional_storage_form"); return f ? f.checkValidity() : "form not found"; }')
            debug_selected = await page.evaluate('() => { var r = document.querySelector("input[name=options]:checked"); return r ? r.value : "none"; }')
            raise Exception(
                f'Neither bootbox nor modal visible. '
                f'modals_show={debug_modals}, bootbox={debug_bootbox}, '
                f'checkValidity={debug_validity}, selectedProvider={debug_selected}, '
                f'body={debug_body[:500]}'
            ) from e

await run_pw(_step)

## 「保存」ボタンをクリックして設定を保存する

「機関ストレージ」画面で「NII Storage」のラジオボタンが選択された状態になり、設定が保存されること（対象機関がNII Storage利用状態に戻ること）

In [ ]:
async def _step(page):
    # Save ボタンをクリック
    await page.locator('#osfstorage_save').click()

    # 保存成功メッセージは一瞬しか表示されずすぐページがリロードされるため、
    # モーダルが閉じたこと→ページ安定→ラジオの最終状態で完了を確認する
    osfstorage_modal = page.locator('#osfstorage_modal')
    await expect(osfstorage_modal).to_be_hidden(timeout=transition_timeout)
    await page.wait_for_load_state('load', timeout=transition_timeout)
    await expect(page.locator('#osfstorage')).to_be_checked(timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}